# Astra Chat — from scratch, no other checkpoints (T4, ~35 min)

Trains a conversational Astra **from random init** on the repo's best text data. Nothing is warm-started from the cyber checkpoints — the cyber model is a telemetry-only byte-BPE and cannot converse (it answers `You: hello` with `dsthostsamesrvrate=1.00 ... | EARLY_WARNING ipsweep`).

**Pipeline (2 stages, 1 tokenizer):**

| Stage | Config | Data | Steps | Output |
|---|---|---|---|---|
| 0 | — | prose + chat | — | `tokenizer/artifacts/prose_chat_word.json` (16k word-level) |
| 1 | `configs/astra5m_word_prose.json` | `datasets/prose/train.txt` (853k words, Gutenberg) | 5400 (~13 epochs) | `/content/runs/word_prose/final.npz` |
| 2 | `configs/astra5m_word_chat.json` | `datasets/chat/train.txt` (913k words, `You:`/`Astra:` pairs) | 3800 (~8.5 epochs) | `/content/runs/word_chat/resumed/final.npz` |

Same 5.1M-param model in both stages (6 layers, d_model 256, 8 heads, seq 256, tied embeddings) — only the corpus changes, so stage 2 resumes stage 1's weights with the *same* tokenizer.

**Budget:** 2048 words/step (batch_seq 8 x seq 256) → 11.1M words in stage 1, 7.8M in stage 2. Expect ~15-20 min + ~10-15 min on a T4.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
import os, subprocess
REPO = '/content/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('repo ready at', os.getcwd())

---
## STAGE 0 — Build the word-level tokenizer (~1 min)

16,384 word-level vocabulary fitted on chat + prose together, so the `You:` / `Astra:` framing and English both round-trip exactly. Emits `tokenizer/artifacts/prose_chat_word.json` and prints a coverage report.

In [ ]:
import os, subprocess
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    subprocess.run('python tools/build_word_tokenizer.py', shell=True, check=True)
else:
    print('word tokenizer already present')
import sys, json
sys.path.insert(0, '/content/astra/python')
from astra.tokenizer.word import WordLevel
tok = WordLevel.load('tokenizer/artifacts/prose_chat_word.json')
print('word vocab:', len(tok))
s = "You: Hello, my name is Astra.\nAstra: Nice to meet you!"
print('round-trip OK:', tok.decode(tok.encode(s)) == s)

---
## STAGE 1 — Base language model on English prose (5400 steps, ~15-20 min)

Learns English grammar and word-level structure from Project Gutenberg (public domain). Ends with **`[step 5400]`**; the final weights land in `/content/runs/word_prose/final.npz`.

In [ ]:
import os, subprocess
os.chdir('/content/astra')
assert os.path.exists('tokenizer/artifacts/prose_chat_word.json'), 'NO TOKENIZER - run STAGE 0 first'
!mkdir -p /content/runs
!python training/gpu_train.py --config configs/astra5m_word_prose.json --out /content/runs/word_prose

---
## STAGE 2 — Chat fine-tune on `You:` / `Astra:` dialogue (3800 steps, ~10-15 min)

Resumes stage 1's weights (same tokenizer, same architecture) and trains on the DailyDialog-derived corpus in `datasets/chat`. This is a warm start from *this notebook's own* stage 1 — no external checkpoint is read.

**Only run after STAGE 1 finished.** Ends with **`[step 3800]`**; final weights land in `/content/runs/word_chat/resumed/final.npz`.

In [ ]:
import os, subprocess, glob
os.chdir('/content/astra')
BASE = '/content/runs/word_prose/final.npz'
assert os.path.exists(BASE), 'NO PROSE FINAL - run STAGE 1 first'
!python training/gpu_train.py --config configs/astra5m_word_chat.json --out /content/runs/word_chat --resume /content/runs/word_prose/final.npz

---
## STAGE 3 — Talk to it

Greedy-by-default decoding with a KV cache reused across turns, so the model keeps the conversation in context. `rep_penalty=1.15` blocks the `Astra: Astra: Astra:` loop small LMs fall into, and `forbidden` masks the special tokens so they never leak into the text. Generation stops at the next `You:` turn.

In [ ]:
import sys, os, glob, json, numpy as np
sys.path.insert(0, '/content/astra/python')
from astra.inference import KVCache, decode
from astra.learning.evaluate import load_model
from astra.model import ModelConfig
from astra.tokenizer.word import WordLevel

os.chdir('/content/astra')
CANDIDATES = sorted(glob.glob('/content/runs/word_chat/**/final.npz', recursive=True))
CKPT = CANDIDATES[0] if CANDIDATES else None
assert CKPT, 'NO CHAT FINAL - run STAGE 2 first'

cfg = json.load(open('configs/astra5m_word_chat.json'))
tok = WordLevel.load(cfg['tokenizer'])
mcfg = ModelConfig.from_dict(cfg['model']); mcfg.vocab_size = len(tok)
model = load_model(mcfg, CKPT)
cache = KVCache(mcfg)
history = []
SPECIALS = set(range(tok.num_special))  # <pad> <bos> <eos> <unk> are ids 0..3
print('checkpoint:', CKPT, '| vocab:', len(tok), '| window:', mcfg.max_seq_len)

def ask(user_text, max_new=48, temperature=0.7, top_k=40, rep_penalty=1.15):
    global history
    history.append(f'You: {user_text}')
    prompt = '\n'.join(history) + '\nAstra:'
    ids = tok.encode(prompt)[-(mcfg.max_seq_len - max_new):]
    new = decode(model, ids, max_new=max_new, temperature=temperature,
                 rng=np.random.default_rng(len(history)), top_k=top_k,
                 cache=cache, rep_penalty=rep_penalty, forbidden=SPECIALS)
    reply = tok.decode(new).split('\nYou:')[0].strip()
    history.append(f'Astra: {reply}')
    return reply

for q in ['hello', 'what is coding', 'why do leaves fall', 'tell me a joke']:
    print('You:', q)
    print('Astra:', ask(q), '\n')

In [ ]:
# Interactive loop (works in Colab; cancel the kernel to exit)
while True:
    try:
        q = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print()
        break
    if not q or q.lower() in {'quit', 'exit', 'bye'}:
        break
    print('Astra:', ask(q))

---
## STAGE 4 — Keep the result

Saves the base and chat weights to Drive under distinct names, then offers both as downloads.

In [ ]:
import os, glob, shutil, hashlib
from google.colab import drive, files

drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/astra_checkpoints'
os.makedirs(DEST, exist_ok=True)

def newest(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[-1] if hits else None

OUT = {
    'astra_word_prose_final.npz': newest('/content/runs/word_prose/**/final.npz'),
    'astra_word_chat_final.npz': newest('/content/runs/word_chat/**/final.npz'),
}
for name, src in OUT.items():
    if not src:
        print(name, 'NOT PRESENT')
        continue
    shutil.copy(src, f'/content/{name}')
    shutil.copy(src, os.path.join(DEST, name))
    h = hashlib.md5(open(src, 'rb').read()).hexdigest()[:12]
    print(name, os.path.getsize(src), 'md5', h)
    files.download(f'/content/{name}')